# BERTweet Binary Slang Classifier

Fine-tunes `vinai/bertweet-base` with a `[CLS]`-based binary classification head for sentence-level slang detection (`SLANG` vs `NOT_SLANG`).

**Pipeline stages:**
1. Hyperparameter sweep (lr × batch size × max_seq_length × epochs)
2. Dataset size ablation (25 / 50 / 75 / 100% of training data)
3. Final evaluation on test and generalization_test splits

Results are saved to `bertweet_binary_results/`.

## Setup

In [1]:
# Install dependencies
!pip install torch>=2.2.0 transformers>=4.40.0 accelerate>=0.29.0 datasets>=2.18.0 \
    scikit-learn>=1.4.0 seqeval>=1.2.2 pandas>=2.0.0 numpy>=1.26.0 "emoji==0.6.0" -q

In [6]:
# Optional: mount Google Drive to persist results across sessions
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import csv
import itertools
import json
import os

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

print(f"PyTorch {torch.__version__}  |  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch 2.10.0+cu128  |  CUDA available: True
GPU: NVIDIA A100-SXM4-80GB


## Configuration

Edit the paths below if your data files are in a different location (e.g. on Google Drive).

In [18]:
# --- Data paths ---
TRAIN_PATH          = "/content/drive/MyDrive/CSCI 544 Group Project/output/train.csv"
DEV_PATH            = "/content/drive/MyDrive/CSCI 544 Group Project/output/dev.csv"
TEST_PATH           = "/content/drive/MyDrive/CSCI 544 Group Project/output/test.csv"
GENERALIZATION_PATH = "/content/drive/MyDrive/CSCI 544 Group Project/output/generalization_test.csv"

# --- Output directory (change to a Drive path to persist across sessions) ---
OUTPUT_DIR = "/content/drive/MyDrive/CSCI 544 Group Project/bertweet_binary_results"
# OUTPUT_DIR = "bertweet_binary_results"

# --- Pipeline control ---
SWEEP_ONLY    = False   # Set True to stop after sweep and save best_config.json
SKIP_SWEEP    = False   # Set True to skip sweep and use BEST_CONFIG below
SKIP_ABLATION = False   # Set True to skip ablation

# Used when SKIP_SWEEP = True
BEST_CONFIG = {
    "learning_rate": 3e-5,
    "per_device_train_batch_size": 32,
    "max_seq_length": 128,
    "num_train_epochs": 3,
}

# --- Model ---
MODEL_NAME  = "vinai/bertweet-base"
RANDOM_SEED = 42

# --- Labels ---
NOT_SLANG = "NOT_SLANG"
SLANG     = "SLANG"
LABEL_MAP = {SLANG: 1, NOT_SLANG: 0}
ID2LABEL  = {0: NOT_SLANG, 1: SLANG}
LABEL2ID  = {NOT_SLANG: 0, SLANG: 1}

# --- Sweep grid (3 × 2 × 2 × 2 = 24 configs) ---
SWEEP_GRID = {
    "learning_rate":               [2e-5, 3e-5, 5e-5],
    "per_device_train_batch_size": [16, 32],
    "max_seq_length":              [64, 128],
    "num_train_epochs":            [3, 5],
}

ABLATION_FRACTIONS = [0.25, 0.50, 0.75, 1.00]

os.makedirs(OUTPUT_DIR, exist_ok=True)

## Dataset, Model Helpers, and Training Utilities

In [19]:
class SlangDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=128):
        self.texts      = df["text"].fillna("").astype(str).tolist()
        self.labels     = [LABEL_MAP[l] for l in df["label"].tolist()]
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        # BERTweet has no segment embeddings — do not include token_type_ids
        return {
            "input_ids":      encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels":         torch.tensor(self.labels[idx], dtype=torch.long),
        }

In [20]:
def load_data(path):
    return pd.read_csv(path)


def fresh_model():
    """Load a fresh copy of BERTweet with a binary classification head."""
    return AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
    )


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy":  accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall":    recall_score(labels, preds, zero_division=0),
        "f1":        f1_score(labels, preds, zero_division=0),
    }


def get_training_args(config, run_dir, do_save=False):
    return TrainingArguments(
        output_dir=run_dir,
        num_train_epochs=config["num_train_epochs"],
        per_device_train_batch_size=config["per_device_train_batch_size"],
        per_device_eval_batch_size=64,
        learning_rate=config["learning_rate"],
        warmup_ratio=0.1,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch" if do_save else "no",
        load_best_model_at_end=do_save,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_strategy="epoch",
        report_to="none",
        seed=RANDOM_SEED,
        fp16=torch.cuda.is_available(),
        dataloader_num_workers=0,
    )


def _cleanup(model, trainer):
    """Explicitly free GPU memory after each training run."""
    del model, trainer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [21]:
def run_sweep(train_df, dev_df, tokenizer):
    configs = list(itertools.product(
        SWEEP_GRID["learning_rate"],
        SWEEP_GRID["per_device_train_batch_size"],
        SWEEP_GRID["max_seq_length"],
        SWEEP_GRID["num_train_epochs"],
    ))
    records = []
    best_f1, best_config = -1.0, None

    for i, (lr, bs, seq_len, epochs) in enumerate(configs):
        config = {
            "learning_rate":               lr,
            "per_device_train_batch_size": bs,
            "max_seq_length":              seq_len,
            "num_train_epochs":            epochs,
        }
        run_dir = os.path.join(OUTPUT_DIR, "checkpoints", f"run_{i:02d}")
        print(f"\n[Sweep {i + 1}/{len(configs)}] {config}")

        train_ds = SlangDataset(train_df, tokenizer, seq_len)
        dev_ds   = SlangDataset(dev_df,   tokenizer, seq_len)

        model   = fresh_model()
        t_args  = get_training_args(config, run_dir, do_save=False)
        trainer = Trainer(
            model=model, args=t_args,
            train_dataset=train_ds, eval_dataset=dev_ds,
            compute_metrics=compute_metrics,
        )
        trainer.train()
        metrics = trainer.evaluate()

        record = {**config, **{k.replace("eval_", ""): v for k, v in metrics.items()}}
        records.append(record)
        print(f"  Dev F1={metrics.get('eval_f1', 0):.4f}  Acc={metrics.get('eval_accuracy', 0):.4f}")

        if metrics.get("eval_f1", 0) > best_f1:
            best_f1     = metrics["eval_f1"]
            best_config = config.copy()

        _cleanup(model, trainer)

    print(f"\nBest dev F1: {best_f1:.4f}  Config: {best_config}")
    return best_config, records

In [22]:
def run_ablation(train_df, dev_df, test_df, tokenizer, best_config):
    records = []
    seq_len = best_config["max_seq_length"]

    for frac in ABLATION_FRACTIONS:
        if frac < 1.0:
            subset_df = (
                train_df.groupby("label", group_keys=False)
                .apply(lambda g: g.sample(frac=frac, random_state=RANDOM_SEED))
                .reset_index(drop=True)
            )
        else:
            subset_df = train_df

        n_train = len(subset_df)
        run_dir = os.path.join(OUTPUT_DIR, "checkpoints", f"ablation_frac{int(frac * 100):03d}")
        print(f"\n[Ablation] frac={frac:.2f}  n_train={n_train}")

        train_ds = SlangDataset(subset_df, tokenizer, seq_len)
        dev_ds   = SlangDataset(dev_df,    tokenizer, seq_len)
        test_ds  = SlangDataset(test_df,   tokenizer, seq_len)

        model   = fresh_model()
        t_args  = get_training_args(best_config, run_dir, do_save=False)
        trainer = Trainer(
            model=model, args=t_args,
            train_dataset=train_ds, eval_dataset=dev_ds,
            compute_metrics=compute_metrics,
        )
        trainer.train()

        dev_metrics  = trainer.evaluate(dev_ds)
        test_metrics = trainer.evaluate(test_ds)

        record = {
            "fraction": frac,
            "n_train":  n_train,
            **{f"dev_{k.replace('eval_', '')}":  v for k, v in dev_metrics.items()},
            **{f"test_{k.replace('eval_', '')}": v for k, v in test_metrics.items()},
        }
        records.append(record)
        print(f"  Dev F1={dev_metrics.get('eval_f1', 0):.4f}  Test F1={test_metrics.get('eval_f1', 0):.4f}")

        _cleanup(model, trainer)

    return records

In [23]:
def run_final_eval(train_df, dev_df, test_df, gen_df, tokenizer, best_config):
    seq_len = best_config["max_seq_length"]
    run_dir = os.path.join(OUTPUT_DIR, "checkpoints", "final")
    print(f"\n[Final Eval] Training with best config: {best_config}")

    train_ds = SlangDataset(train_df, tokenizer, seq_len)
    dev_ds   = SlangDataset(dev_df,   tokenizer, seq_len)
    test_ds  = SlangDataset(test_df,  tokenizer, seq_len)
    gen_ds   = SlangDataset(gen_df,   tokenizer, seq_len)

    model   = fresh_model()
    t_args  = get_training_args(best_config, run_dir, do_save=True)
    trainer = Trainer(
        model=model, args=t_args,
        train_dataset=train_ds, eval_dataset=dev_ds,
        compute_metrics=compute_metrics,
    )
    trainer.train()

    test_metrics = trainer.evaluate(test_ds)
    gen_metrics  = trainer.evaluate(gen_ds)

    print(f"  Test F1={test_metrics.get('eval_f1', 0):.4f}")
    print(f"  Gen  F1={gen_metrics.get('eval_f1', 0):.4f}")
    return test_metrics, gen_metrics

In [29]:
def save_sweep_results(records, path):
    if not records:
        return
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(records[0].keys()))
        writer.writeheader()
        writer.writerows(records)
    print(f"Sweep results saved to {path}")


def save_ablation_results(records, path):
    if not records:
        return
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(records[0].keys()))
        writer.writeheader()
        writer.writerows(records)
    print(f"Ablation results saved to {path}")


def save_final_report(test_metrics, gen_metrics, best_config, path):
    os.makedirs(os.path.dirname(path) if os.path.dirname(path) else ".", exist_ok=True)

    def m(metrics, key):
        return metrics.get(f"eval_{key}", metrics.get(key, float("nan")))

    test_f1 = m(test_metrics, "f1")
    gen_f1  = m(gen_metrics,  "f1")

    with open(path, "w", encoding="utf-8") as f:
        f.write("=" * 60 + "\n")
        f.write("BERTWEET BINARY FINE-TUNING REPORT\n")
        f.write("=" * 60 + "\n\n")

        f.write("=== BEST HYPERPARAMETERS ===\n")
        f.write(f"  Learning Rate             : {best_config['learning_rate']}\n")
        f.write(f"  Batch Size                : {best_config['per_device_train_batch_size']}\n")
        f.write(f"  Max Sequence Length       : {best_config['max_seq_length']}\n")
        f.write(f"  Num Epochs                : {best_config['num_train_epochs']}\n\n")

        f.write("=== TEST SET RESULTS ===\n")
        f.write(f"  Accuracy  : {m(test_metrics, 'accuracy'):.4f}\n")
        f.write(f"  Precision : {m(test_metrics, 'precision'):.4f}\n")
        f.write(f"  Recall    : {m(test_metrics, 'recall'):.4f}\n")
        f.write(f"  F1        : {m(test_metrics, 'f1'):.4f}\n\n")

        f.write("=== GENERALIZATION TEST RESULTS ===\n")
        f.write(f"  Accuracy  : {m(gen_metrics, 'accuracy'):.4f}\n")
        f.write(f"  Precision : {m(gen_metrics, 'precision'):.4f}\n")
        f.write(f"  Recall    : {m(gen_metrics, 'recall'):.4f}\n")
        f.write(f"  F1        : {m(gen_metrics, 'f1'):.4f}\n\n")

        f.write("=== COMPARISON TO BASELINES ===\n")
        f.write(f"  {'Model':<30} {'Test F1':>10}  {'Gen Test F1':>12}\n")
        f.write("  " + "-" * 56 + "\n")
        f.write(f"  {'Dictionary Baseline':<30} {'0.6770':>10}  {'0.7437':>12}\n")
        f.write(f"  {'TF-IDF + LogReg':<30} {'0.8472':>10}  {'0.8796':>12}\n")
        f.write(f"  {'BERTweet (this run)':<30} {test_f1:>10.4f}  {gen_f1:>12.4f}\n")

    print(f"Final report saved to {path}")

## Load Tokenizer and Data

In [25]:
# use_fast=False: BERTweet BPE slow tokenizer required for normalization=True
# normalization=True: applies TweetNormalizer (URLs->HTTPURL, @mentions->@USER)
print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False, normalization=True)

print("Loading data...")
train_df = load_data(TRAIN_PATH)
dev_df   = load_data(DEV_PATH)
test_df  = load_data(TEST_PATH)
gen_df   = load_data(GENERALIZATION_PATH)
print(f"  train={len(train_df)}  dev={len(dev_df)}  test={len(test_df)}  gen={len(gen_df)}")

Loading tokenizer: vinai/bertweet-base
Loading data...
  train=14000  dev=3000  test=3000  gen=1000


## Step 1: Hyperparameter Sweep

Grid: `learning_rate` × `batch_size` × `max_seq_length` × `num_epochs` = **24 configs**.

Best config is selected by dev F1 and saved to `bertweet_binary_results/best_config.json`.

Set `SWEEP_ONLY = True` in the Configuration cell to stop here and resume later with `SKIP_SWEEP = True`.

In [26]:
if not SKIP_SWEEP:
    print("=" * 60)
    print("HYPERPARAMETER SWEEP")
    print("=" * 60)
    best_config, sweep_records = run_sweep(train_df, dev_df, tokenizer)
    save_sweep_results(sweep_records, os.path.join(OUTPUT_DIR, "sweep_results.csv"))
    best_config_path = os.path.join(OUTPUT_DIR, "best_config.json")
    with open(best_config_path, "w", encoding="utf-8") as f:
        json.dump(best_config, f, indent=2)
    print(f"Best config saved to {best_config_path}")
else:
    best_config = BEST_CONFIG
    print(f"Skipping sweep. Using config: {best_config}")

if SWEEP_ONLY:
    print("\nSWEEP_ONLY=True — stopping here.")
    print(f"To resume, set SKIP_SWEEP=True and BEST_CONFIG={best_config}")

HYPERPARAMETER SWEEP

[Sweep 1/24] {'learning_rate': 2e-05, 'per_device_train_batch_size': 16, 'max_seq_length': 64, 'num_train_epochs': 3}


pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/bertweet-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.decoder.weight          | UNEXPECTED | 
roberta.pooler.dense.weight     | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.pooler.dense.bias       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initi

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

...

  Dev F1=0.9062  Acc=0.9047

[Sweep 24/24] {'learning_rate': 5e-05, 'per_device_train_batch_size': 32, 'max_seq_length': 128, 'num_train_epochs': 5}


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/bertweet-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.decoder.weight          | UNEXPECTED | 
roberta.pooler.dense.weight     | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.pooler.dense.bias       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initi

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.413638,0.278194,0.893667,0.908086,0.876000,0.891754
2,0.232280,0.269444,0.898000,0.886658,0.912667,0.899474
3,0.141928,0.326035,0.902000,0.877820,0.934000,0.905039
4,0.071893,0.394467,0.903000,0.888746,0.921333,0.904746
5,0.034335,0.497736,0.902000,0.886538,0.922000,0.903922


  Dev F1=0.9039  Acc=0.9020

Best dev F1: 0.9088  Config: {'learning_rate': 2e-05, 'per_device_train_batch_size': 16, 'max_seq_length': 64, 'num_train_epochs': 3}
Sweep results saved to /content/drive/MyDrive/CSCI 544 Group Project/bertweet_binary_results/sweep_results.csv
Best config saved to /content/drive/MyDrive/CSCI 544 Group Project/bertweet_binary_results/best_config.json


## Step 2: Dataset Size Ablation

Trains on 25 / 50 / 75 / 100% of the training data using the best config.
Results saved to `bertweet_binary_results/ablation_results.csv`.

In [27]:
if not SWEEP_ONLY and not SKIP_ABLATION:
    print("=" * 60)
    print("DATASET SIZE ABLATION")
    print("=" * 60)
    ablation_records = run_ablation(train_df, dev_df, test_df, tokenizer, best_config)
    save_ablation_results(ablation_records, os.path.join(OUTPUT_DIR, "ablation_results.csv"))
elif SWEEP_ONLY:
    print("Skipped (SWEEP_ONLY=True).")
else:
    print("Skipped (SKIP_ABLATION=True).")

DATASET SIZE ABLATION

[Ablation] frac=0.25  n_train=3500


/tmp/ipykernel_2520/3151373259.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(frac=frac, random_state=RANDOM_SEED))


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/bertweet-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.decoder.weight          | UNEXPECTED | 
roberta.pooler.dense.weight     | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.pooler.dense.bias       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initi

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.526331,0.387811,0.832333,0.810979,0.866667,0.837899
2,0.307552,0.368578,0.862333,0.843335,0.890000,0.866040
3,0.208559,0.389445,0.872000,0.846584,0.908667,0.876527


...

  Dev F1=0.9010  Test F1=0.9103

[Ablation] frac=1.00  n_train=14000


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/bertweet-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.decoder.weight          | UNEXPECTED | 
roberta.pooler.dense.weight     | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.pooler.dense.bias       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initi

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.408675,0.287551,0.888667,0.892857,0.883333,0.888070
2,0.235373,0.290836,0.897000,0.895681,0.898667,0.897171
3,0.160567,0.346055,0.905667,0.884883,0.932667,0.908147


  Dev F1=0.9081  Test F1=0.9113
Ablation results saved to /content/drive/MyDrive/CSCI 544 Group Project/bertweet_binary_results/ablation_results.csv


## Step 3: Final Evaluation

Trains on full training data with the best config, evaluates on `test` and `generalization_test`.
Best checkpoint saved to `bertweet_binary_results/checkpoints/final/`.
Summary saved to `bertweet_binary_results/final_report.txt`.

In [30]:
if not SWEEP_ONLY:
    print("=" * 60)
    print("FINAL EVALUATION")
    print("=" * 60)
    test_metrics, gen_metrics = run_final_eval(
        train_df, dev_df, test_df, gen_df, tokenizer, best_config
    )
    save_final_report(
        test_metrics, gen_metrics, best_config,
        os.path.join(OUTPUT_DIR, "final_report.txt"),
    )
else:
    print("Skipped (SWEEP_ONLY=True).")

FINAL EVALUATION

[Final Eval] Training with best config: {'learning_rate': 2e-05, 'per_device_train_batch_size': 16, 'max_seq_length': 64, 'num_train_epochs': 3}


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/bertweet-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.decoder.weight          | UNEXPECTED | 
roberta.pooler.dense.weight     | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.pooler.dense.bias       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initi

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.408675,0.287551,0.888667,0.892857,0.883333,0.888070
2,0.235373,0.290836,0.897000,0.895681,0.898667,0.897171
3,0.160567,0.346055,0.905667,0.884883,0.932667,0.908147


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

  Test F1=0.9113
  Gen  F1=0.9545
Final report saved to /content/drive/MyDrive/CSCI 544 Group Project/bertweet_binary_results/final_report.txt
